In [1]:
import nltk
import tensorflow
import warnings
warnings.filterwarnings('ignore')


I0000 00:00:1781929634.655428   41043 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
token_sent = '''hi, today we are gonna perform tokenization using nltk library '''

In [3]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /home/bhavish-
[nltk_data]     berry/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
tokenised_ = nltk.tokenize.sent_tokenize(token_sent)
tokenised_

['hi, today we are gonna perform tokenization using nltk library']

In [5]:
import tensorflow_datasets as tfds
raw_train_set, raw_valid_set,raw_test_set = tfds.load(
    name= 'imdb_reviews',
    split=["train[:90%]","train[90%:]","test"],
    as_supervised=True
)

I0000 00:00:1781929637.231592   41043 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2143 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [6]:
import tensorflow as tf
tf.random.set_seed(42)
train_set = raw_train_set.shuffle(5000,seed=42).batch(16).prefetch(1)
valid_set = raw_valid_set.batch(16).prefetch(1)
test_set = raw_test_set.batch(16).prefetch(1)

In [7]:
import numpy
for review,label in raw_test_set.take(2):
    print(review.numpy().decode('utf-8'))
    print("label:", label.numpy())


There are films that make careers. For George Romero, it was NIGHT OF THE LIVING DEAD; for Kevin Smith, CLERKS; for Robert Rodriguez, EL MARIACHI. Add to that list Onur Tukel's absolutely amazing DING-A-LING-LESS. Flawless film-making, and as assured and as professional as any of the aforementioned movies. I haven't laughed this hard since I saw THE FULL MONTY. (And, even then, I don't think I laughed quite this hard... So to speak.) Tukel's talent is considerable: DING-A-LING-LESS is so chock full of double entendres that one would have to sit down with a copy of this script and do a line-by-line examination of it to fully appreciate the, uh, breadth and width of it. Every shot is beautifully composed (a clear sign of a sure-handed director), and the performances all around are solid (there's none of the over-the-top scenery chewing one might've expected from a film like this). DING-A-LING-LESS is a film whose time has come.
label: 1
A blackly comic tale of a down-trodden priest, Naza

I0000 00:00:1781929638.461167   41211 tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
W0000 00:00:1781929638.467515   41216 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [8]:
vocab_size = 1000
text_vec_layer = tf.keras.layers.TextVectorization(max_tokens=vocab_size)
text_vec_layer.adapt(train_set.map(lambda reviews, labels: reviews))

In [9]:
tf.config.list_physical_devices()

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'),
 PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [15]:
embed_size = 128
tf.random.set_seed(42)
model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1,activation='sigmoid')
])
model.compile(loss='binary_crossentropy',optimizer='nadam',
              metrics=['accuracy'])

In [16]:
history = model.fit(train_set,validation_data=valid_set,epochs=2)

Epoch 1/2
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 17ms/step - accuracy: 0.7546 - loss: 0.4816 - val_accuracy: 0.8624 - val_loss: 0.3175
Epoch 2/2
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 23s 17ms/step - accuracy: 0.8712 - loss: 0.3051 - val_accuracy: 0.8536 - val_loss: 0.3313
